<a href="https://colab.research.google.com/github/K-Irungu/apache-spark-assignment/blob/main/Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Apache Spark Assignment: Chicago Taxi Trips Analysis
**Name:** Kevin Thuita Irungu  
**Student ID:** 668115  
**Date:** 24/09/2026

## Stage 1: Initialize SparkSession and Load Data

### Install pyspark and requests

In [1]:
!pip install pyspark requests

### Download the dataset

In [2]:
import requests

url = "https://data.cityofchicago.org/resource/wrvz-psew.csv?$limit=200000&$order=trip_start_timestamp"
local_path = "chicago_taxi_trips.csv"

response = requests.get(url, timeout=120)
response.raise_for_status()

with open(local_path, "wb") as f:
    f.write(response.content)

print(f"Downloaded file size: {len(response.content) / (1024*1024):.2f} MB")

Downloaded file size: 83.18 MB


### Create SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark import SparkContext

# Stop any existing SparkSession
active_spark_session = SparkSession.getActiveSession()
if active_spark_session:
    print("Stopping existing SparkSession.")
    active_spark_session.stop()

# Ensure no SparkContext is running (more robust check)
if SparkContext._active_spark_context is not None:
    print("Stopping existing SparkContext directly.")
    SparkContext._active_spark_context.stop()

# Now create the new SparkSession
spark = SparkSession.builder \
    .appName("ChicagoTaxiAnalysis") \
    .getOrCreate()

spark

### Load csv into a Spark DataFrame

In [4]:
taxi_df = spark.read.csv(local_path, header=True, inferSchema=True)

### Print counts

In [5]:
row_count = taxi_df.count()
col_count = len(taxi_df.columns)

print(f"Total rows: {row_count}")
print(f"Total columns: {col_count}")

Total rows: 200000
Total columns: 23


## Stage 2: Inspect the DataFrame

### Print the chema

In [6]:
taxi_df.printSchema()

root
 |-- trip_id: string (nullable = true)
 |-- taxi_id: string (nullable = true)
 |-- trip_start_timestamp: timestamp (nullable = true)
 |-- trip_end_timestamp: timestamp (nullable = true)
 |-- trip_seconds: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- pickup_census_tract: long (nullable = true)
 |-- dropoff_census_tract: long (nullable = true)
 |-- pickup_community_area: integer (nullable = true)
 |-- dropoff_community_area: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- extras: double (nullable = true)
 |-- trip_total: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- company: string (nullable = true)
 |-- pickup_centroid_latitude: double (nullable = true)
 |-- pickup_centroid_longitude: double (nullable = true)
 |-- pickup_centroid_location: string (nullable = true)
 |-- dropoff_centroid_latitude: double (nullable = true)
 |-- dropoff_cen

### List all column names

In [7]:
print(taxi_df.columns)

['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras', 'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude', 'pickup_centroid_longitude', 'pickup_centroid_location', 'dropoff_centroid_latitude', 'dropoff_centroid_longitude', 'dropoff_centroid_location']


### Display first 5 rows

In [8]:
taxi_df.show(5, truncate=False)

+----------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+--------------------+-------------------+------------+----------+-------------------+--------------------+---------------------+----------------------+------+-----+-----+------+----------+------------+-------+------------------------+-------------------------+------------------------------------+-------------------------+--------------------------+-----------------------------------+
|trip_id                                 |taxi_id                                                                                                                         |trip_start_timestamp|trip_end_timestamp |trip_seconds|trip_miles|pickup_census_tract|dropoff_census_tract|pickup_community_area|dropoff_community_area|fare  |tips |tolls|extras|trip_total|payment_type|company|pickup_centroid_latitude|pickup_centroid_longitude|pickup_cent

### Dataset Structure Reflection


The dataset contains 23 columns spanning several types: identifiers (`trip_id`, `taxi_id` as strings), timestamps (`trip_start_timestamp`, `trip_end_timestamp`), numeric trip metrics (`trip_seconds` as integer, `trip_miles`, `fare`, `tips`, `tolls`, `extras`, `trip_total` as doubles), geographic references (`pickup_community_area`, `dropoff_community_area` as integers, plus census tracts and centroid latitude/longitude/location fields), and categorical columns (`payment_type`, `company`).


For trip analysis, the most important columns are `trip_start_timestamp`, `trip_seconds`, `trip_miles`, `fare`/`trip_total`, and the pickup/dropoff community areas, since these drive time-based, distance-based, and location-based analysis. `inferSchema=True` correctly detected `trip_seconds` as integer and the timestamp columns as proper timestamp type without needing manual casting, and the sample rows show several trips have `NULL` values for census tract, community area, and company fields.

## Stage 3: Identify and Handle Missing Data

### Count nulls per column

In [12]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = taxi_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in taxi_df.columns
])

null_counts.show(vertical=True, truncate=False)

-RECORD 0----------------------------
 trip_id                    | 0      
 taxi_id                    | 0      
 trip_start_timestamp       | 0      
 trip_end_timestamp         | 0      
 trip_seconds               | 3101   
 trip_miles                 | 5      
 pickup_census_tract        | 93490  
 dropoff_census_tract       | 100340 
 pickup_community_area      | 30354  
 dropoff_community_area     | 40599  
 fare                       | 0      
 tips                       | 0      
 tolls                      | 0      
 extras                     | 0      
 trip_total                 | 0      
 payment_type               | 0      
 company                    | 93843  
 pickup_centroid_latitude   | 30354  
 pickup_centroid_longitude  | 30354  
 pickup_centroid_location   | 30354  
 dropoff_centroid_latitude  | 40581  
 dropoff_centroid_longitude | 40581  
 dropoff_centroid_location  | 40581  



### Drop rows missing essential metrics

In [13]:
taxi_df = taxi_df.dropna(subset=["trip_total", "trip_miles"])

### Fill nulls in payment_type with UNKNOWN

In [14]:
taxi_df = taxi_df.fillna({"payment_type": "UNKNOWN"})

### Print new row count

In [15]:
new_row_count = taxi_df.count()
print(f"Row count after cleaning: {new_row_count}")

Row count after cleaning: 199995


### Missing Data Reflection

The columns with the highest null counts were `pickup_census_tract`, `dropoff_census_tract`, and `company`. This is because census tract data is only recorded when GPS coordinates fall within a mapped tract boundary, and many trips - especially those picked up or dropped off in areas without precise geolocation data, or trips logged by taxi companies that don't report a `company` name to the city — leave these fields blank. `pickup_community_area` and `dropoff_community_area` also had a notable number of nulls, likely for the same geolocation-reporting reasons.


In contrast, core trip metrics like `trip_total`, `trip_miles`, and `fare` had far fewer nulls, since these are essential, directly metered values that are almost always captured during the trip itself.